In [ ]:
import orjson
import genson
import natsort
import thefuzz.fuzz
import thefuzz.process
import rich
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig

In [ ]:
# --- Sinks config ---
data_index_catalog_config = S3TablesCatalogConfig(
    region="ap-southeast-2",
    arn="arn:aws:s3tables:ap-southeast-2:704910415367:bucket/data-index",
)

structured_table = IcebergTableConfig(
    catalog_config=data_index_catalog_config,
    namespace="data_index",
    table_name=f"structured_metadata_v1",
).load()

unstructured_table = IcebergTableConfig(
    catalog_config=data_index_catalog_config,
    namespace="data_index",
    table_name="unstructured_metadata",
).load()

In [ ]:
# Read into memory as polars.DataFrame
sdf = structured_table.scan().to_polars()
udf = unstructured_table.scan().to_polars()
display(sdf)
display(udf)

In [ ]:
# Construct generator
metadata = (orjson.loads(metadata) for metadata in udf["metadata"])

# Build schema
schema_builder = genson.SchemaBuilder()
for metadata in metadata:
    schema_builder.add_object(metadata)

# Extract schema
schema = schema_builder.to_schema()

In [ ]:
# Sort global attribute keys naturally (human-like)
sorted_global_attribute_keys = natsort.natsorted(
    seq=[k for k in schema["properties"]["global_attrs"]["properties"]],
    alg=natsort.IGNORECASE,
)
print(sorted_global_attribute_keys)

In [ ]:
# Tuning parameters
SIMILARITY_THRESHOLD = 80
# Using token_sort_ratio handles underscores and word order swaps well
SCORER = thefuzz.fuzz.token_sort_ratio 

clusters = []
visited = set()

for key in sorted_global_attribute_keys:
    if key in visited:
        continue
    
    # Find all items in the list that match the current key above our threshold
    matches = thefuzz.process.extractBests(
        query=key, 
        choices=sorted_global_attribute_keys, 
        scorer=SCORER, 
        score_cutoff=SIMILARITY_THRESHOLD
    )
    
    # Filter matches to only include keys we haven't clustered yet
    current_cluster = [match[0] for match in matches if match[0] not in visited]
    
    if current_cluster:
        clusters.append(current_cluster)
        # Mark all keys in this cluster as visited so they aren't processed again
        visited.update(current_cluster)

# Print the resulting list of lists
import pprint
pprint.pprint(clusters)